# MeshPort · Colab A100

Run all cells, then paste the complete URL printed by the final cell into the localhost app. Upload one image or up to six views of the same object in localhost. Generation runs here; preview and textured FBX conversion run locally.

Select an A100 GPU before running. First setup and downloads can take several minutes. The tunnel requires a random session key included in the URL fragment. Treat the complete URL as private; notebook outputs may contain that key. Clear outputs before sharing the notebook. Keep Colab connected.

Uses TRELLIS-image-large with seed 42, 16 sampling steps and 2048 textures. For multiple inputs, TRELLIS blends each view at every geometry and appearance step. This requires more GPU time as the number of views grows. Multi-image conditioning is experimental.


In [ ]:
# @title 0. Verify runtime before setup
import sys, subprocess, pathlib
print("Colab kernel Python:", sys.version)
for cmd in [["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], ["nvcc", "--version"]]:
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout, result.stderr)
print("Existing TRELLIS:", pathlib.Path("/content/TRELLIS").exists())
print("Input images:", [str(p) for p in pathlib.Path("/content").glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}])


In [ ]:
# @title 1. Install an isolated TRELLIS environment (rerunnable)
import os, pathlib, subprocess, textwrap
setup_script = r'''
set -euo pipefail
cd /content
export MAMBA_ROOT_PREFIX=/content/mamba
if [ ! -x /content/bin/micromamba ]; then
  curl -fLs https://micro.mamba.pm/api/micromamba/linux-64/latest -o /content/micromamba.tar.bz2
  tar -xjf /content/micromamba.tar.bz2 -C /content bin/micromamba
fi
if [ ! -x /content/trellis-env/bin/python ]; then
  /content/bin/micromamba create -y -p /content/trellis-env -c conda-forge -c nvidia python=3.10 pip cuda-nvcc=12.1 cuda-cudart-dev=12.1 cuda-cccl=12.1 gcc_linux-64=12 gxx_linux-64=12
fi
export PATH=/content/trellis-env/bin:$PATH
export CUDA_HOME=/content/trellis-env
export CC=/content/trellis-env/bin/x86_64-conda-linux-gnu-cc
export CXX=/content/trellis-env/bin/x86_64-conda-linux-gnu-c++
export TORCH_CUDA_ARCH_LIST=8.0 MAX_JOBS=4
export ATTN_BACKEND=xformers SPCONV_ALGO=native
if [ ! -d /content/TRELLIS/.git ]; then
  git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
fi
cd /content/TRELLIS
git rev-parse HEAD
printf 'numpy==1.26.4
scipy==1.14.1
torch==2.4.0
torchvision==0.19.0
xformers==0.0.27.post2
pymeshfix==0.17.0
pyvista==0.44.2
vtk==9.3.1
transformers==4.46.3
huggingface-hub==0.25.2
fastapi==0.115.6
pydantic==2.10.6
starlette==0.41.3
' > /content/trellis-constraints.txt
export PIP_CONSTRAINT=/content/trellis-constraints.txt
python -m pip install --disable-pip-version-check 'setuptools<70' wheel ninja packaging
python -m pip install torch==2.4.0 torchvision==0.19.0 xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121
python -m pip install numpy==1.26.4 scipy==1.14.1 pillow imageio imageio-ffmpeg tqdm easydict opencv-python-headless rembg==2.0.61 onnxruntime trimesh open3d xatlas pyvista==0.44.2 pymeshfix==0.17.0 igraph transformers==4.46.3 huggingface-hub==0.25.2 spconv-cu120 gradio==4.44.1 fastapi==0.115.6 pydantic==2.10.6 starlette==0.41.3
python -m pip install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
python -m pip install kaolin==0.17.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html
export CPATH=/content/trellis-env/targets/x86_64-linux/include
for cuda_include in /content/trellis-env/lib/python3.10/site-packages/nvidia/*/include; do
  export CPATH="$CPATH:$cuda_include"
done
export LIBRARY_PATH=/content/trellis-env/targets/x86_64-linux/lib
mkdir -p /content/trellis-extensions
if [ ! -d /content/trellis-extensions/nvdiffrast/.git ]; then
  git clone --branch v0.3.3 https://github.com/NVlabs/nvdiffrast.git /content/trellis-extensions/nvdiffrast
fi
if [ ! -d /content/trellis-extensions/mip-splatting/.git ]; then
  git clone --recursive https://github.com/autonomousvision/mip-splatting.git /content/trellis-extensions/mip-splatting
fi
if ! python -c 'import nvdiffrast' >/dev/null 2>&1; then
  python -m pip install --no-build-isolation /content/trellis-extensions/nvdiffrast
fi
if ! python -c 'import diff_gaussian_rasterization' >/dev/null 2>&1; then
  python -m pip install --no-build-isolation /content/trellis-extensions/mip-splatting/submodules/diff-gaussian-rasterization
fi
python -m pip install numba
python -m pip check
python -u -c "import torch,numpy,scipy,kaolin,nvdiffrast.torch,diff_gaussian_rasterization; from trellis.pipelines import TrellisImageTo3DPipeline; from trellis.utils import postprocessing_utils; assert torch.cuda.is_available(); nvdiffrast.torch.RasterizeCudaContext(); print('VERIFIED',torch.__version__,numpy.__version__,scipy.__version__,torch.cuda.get_device_name(0))"
printf '
TRELLIS dependency imports passed. Run Cell 2.
'
'''
pathlib.Path('/content/trellis-setup.sh').write_text(textwrap.dedent(setup_script))
with open('/content/trellis-setup.log', 'a') as log:
    proc = subprocess.Popen(['bash', '/content/trellis-setup.sh'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line); log.flush()
        print(line, end='')
    rc = proc.wait()
if rc:
    raise RuntimeError(f'Setup stopped with exit code {rc}; see /content/trellis-setup.log. Completed packages are reusable.')


In [ ]:
# @title 2. Start the image-to-3D service
import os, pathlib, subprocess, secrets, time, urllib.request
service_source = '"""Authenticated Colab service, embedded into the notebook by build_notebook.py."""\nimport os, io, base64, json, uuid, time, threading, hashlib\nfrom pathlib import Path\nos.environ[\'ATTN_BACKEND\']=\'xformers\'\nos.environ[\'SPCONV_ALGO\']=\'native\'\nimport torch\nfrom PIL import Image, ImageOps\nfrom fastapi import FastAPI, Request, HTTPException\nfrom fastapi.responses import FileResponse, JSONResponse\nfrom trellis.pipelines import TrellisImageTo3DPipeline\nfrom trellis.utils import postprocessing_utils\nfrom photo_finish import refine, unlit\n\nTOKEN=os.environ[\'STUDIO_KEY\']\nROOT=Path(\'/content/trellis-results\'); ROOT.mkdir(exist_ok=True)\npipeline=TrellisImageTo3DPipeline.from_pretrained(\'microsoft/TRELLIS-image-large\')\npipeline.cuda()\napp=FastAPI(docs_url=None,redoc_url=None,openapi_url=None)\njobs={}; lock=threading.Lock()\nImage.MAX_IMAGE_PIXELS=16_000_000\nMULTI_VIEW_MODE=\'multidiffusion\'\nSIMPLIFY_RATIO = float(os.environ.get(\'MESHPORT_SIMPLIFY_RATIO\', \'0.50\'))\nif not 0 <= SIMPLIFY_RATIO < 1:\n    raise ValueError(\'MESHPORT_SIMPLIFY_RATIO must be in [0, 1).\')\n\n@app.middleware(\'http\')\nasync def auth(request,call_next):\n    import secrets\n    if not secrets.compare_digest(request.headers.get(\'authorization\',\'\'),\'Bearer \'+TOKEN):\n        return JSONResponse({\'detail\':\'Invalid connection key\'},401)\n    return await call_next(request)\n\n@app.get(\'/health\')\ndef health():\n    return {\'protocol\':\'form-studio-v1\',\'ready\':True,\'labeled_cameras\':True,\'model\':\'TRELLIS image large\',\'gpu\':torch.cuda.get_device_name(0),\'steps\':16,\'texture_size\':2048,\'multi_view_conditioning\':f\'multi-image-{MULTI_VIEW_MODE}\'}\n\ndef generate(job_id,images,labels=None):\n    job=jobs[job_id]; start=time.monotonic(); folder=ROOT/job_id\n    try:\n        folder.mkdir()\n        digests=[]\n        for i,image in enumerate(images):\n            path=folder/f\'input-{i+1}.png\';image.save(path);digests.append(hashlib.sha256(path.read_bytes()).hexdigest())\n        params=dict(num_samples=1,seed=42,formats=[\'gaussian\',\'mesh\'],sparse_structure_sampler_params={\'steps\':16,\'cfg_strength\':7.5},slat_sampler_params={\'steps\':16,\'cfg_strength\':3.0})\n        if len(images)==1:\n            job[\'stage\']=\'Generating geometry and appearance from one reference image\'\n            outputs=pipeline.run(images[0],**params)\n            conditioning=\'single-image\'\n        else:\n            job[\'stage\']=f\'Blending {len(images)} views for geometry and appearance\'\n            samplers=(pipeline.sparse_structure_sampler,pipeline.slat_sampler)\n            original_methods=[sampler._inference_model for sampler in samplers]\n            try:\n                outputs=pipeline.run_multi_image(images,mode=MULTI_VIEW_MODE,**params)\n            except Exception:\n                # TRELLIS can leave its injected sampler installed when sampling raises.\n                for sampler,original in zip(samplers,original_methods):\n                    if \'_old_inference_model\' in vars(sampler):\n                        sampler._inference_model=original\n                        del sampler._old_inference_model\n                raise\n            conditioning=f\'multi-image-{MULTI_VIEW_MODE}\'\n        job[\'stage\']=\'Baking 2048 texture\'\n        source_mesh = outputs[\'mesh\'][0]\n        source_faces = len(source_mesh.faces)\n        mesh=postprocessing_utils.to_glb(outputs[\'gaussian\'][0],source_mesh,simplify=SIMPLIFY_RATIO,texture_size=2048)\n        job[\'stage\']=\'Matching reference cameras and applying visible photo detail\'\n        finish = refine(mesh, images, labels)\n        mesh.export(folder/\'model.glb\',tree_postprocessor=unlit)\n        details={\'model\':\'microsoft/TRELLIS-image-large\',\'seed\':42,\'steps\':16,\'texture_size\':2048,\'input_count\':len(images),\'conditioning\':conditioning,\'input_sha256\':digests,\'faces\':len(mesh.faces),\'vertices\':len(mesh.vertices),\'watertight\':bool(mesh.is_watertight),\'seconds\':round(time.monotonic()-start,1)}\n        details.update(source_faces=source_faces, simplify_ratio=SIMPLIFY_RATIO, **finish)\n        (folder/\'manifest.json\').write_text(json.dumps(details,indent=2))\n        job.update(status=\'complete\',stage=\'Model ready\',details=details)\n    except Exception as exc:\n        job.update(status=\'failed\',stage=\'Generation failed\',error=str(exc).replace(TOKEN,\'[redacted]\'))\n    finally:\n        job[\'elapsed\']=round(time.monotonic()-start,1)\n        try:\n            torch.cuda.empty_cache()\n        finally:\n            lock.release()\n\n@app.post(\'/jobs\')\nasync def create(request:Request):\n    if not lock.acquire(blocking=False):raise HTTPException(409,\'GPU is busy. Try again after the current model finishes.\')\n    try:\n        raw=bytearray()\n        async for chunk in request.stream():\n            if len(raw)+len(chunk)>32*1024*1024+4096:raise ValueError(\'Images exceed 24 MB total upload limit\')\n            raw.extend(chunk)\n        body=json.loads(raw); encoded=body[\'images\']\n        if not isinstance(encoded,list) or not 1<=len(encoded)<=6:raise ValueError(\'Choose one to six views of one object\')\n        labels=body.get(\'angles\',[\'unknown\']*len(encoded))\n        if not isinstance(labels,list) or len(labels)!=len(encoded) or any(label not in (\'front\',\'rear\',\'left\',\'right\',\'top\',\'bottom\',\'front_three_quarter\',\'rear_three_quarter\',\'front_three_quarter_nose_left\',\'front_three_quarter_nose_right\',\'rear_three_quarter_nose_left\',\'rear_three_quarter_nose_right\',\'oblique\',\'unknown\') for label in labels):\n            raise ValueError(\'Expected one angle per image\')\n        images=[];total=0\n        for index,item in enumerate(encoded,1):\n            data=base64.b64decode(item,validate=True);total+=len(data)\n            if total>24*1024*1024:raise ValueError(\'Images exceed 24 MB total upload limit\')\n            with Image.open(io.BytesIO(data),formats=[\'PNG\',\'JPEG\',\'WEBP\']) as image:\n                if image.width*image.height>16_000_000:raise ValueError(f\'View {index} exceeds 16 megapixels\')\n                if getattr(image,\'n_frames\',1)!=1:raise ValueError(f\'View {index} must be a still image\')\n                images.append(ImageOps.exif_transpose(image).convert(\'RGBA\'))\n        job_id=uuid.uuid4().hex\n        jobs[job_id]={\'id\':job_id,\'status\':\'running\',\'stage\':\'Preparing images\',\'elapsed\':0}\n        threading.Thread(target=generate,args=(job_id,images,labels),daemon=True).start()\n        return {\'id\':job_id}\n    except Exception as exc:\n        lock.release();raise HTTPException(400,str(exc))\n\n@app.get(\'/jobs/{job_id}\')\ndef status(job_id:str):\n    if job_id not in jobs:raise HTTPException(404)\n    return jobs[job_id]\n\n@app.get(\'/jobs/{job_id}/model\')\ndef model(job_id:str):\n    if job_id not in jobs or jobs[job_id][\'status\']!=\'complete\':raise HTTPException(404)\n    return FileResponse(ROOT/job_id/\'model.glb\',media_type=\'model/gltf-binary\')\n\nif __name__==\'__main__\':\n    import uvicorn\n    uvicorn.run(app,host=\'127.0.0.1\',port=7860,access_log=False)\n'
photo_source = '"""Register foreground photos to a generated textured mesh and refine its UV atlas."""\nimport math\n\nimport cv2\nimport numpy as np\nfrom numba import njit\nfrom PIL import Image\n\n\n@njit\ndef raster(screen, faces, height, width):\n    """Nearest triangle and barycentric coordinates at each covered pixel."""\n    owner = np.full((height, width), -1, np.int32)\n    bary = np.zeros((height, width, 2), np.float32)\n    depth = np.full((height, width), -np.inf, np.float32)\n    for fi, (i, j, k) in enumerate(faces):\n        ax, ay, az = screen[i]\n        bx, by, bz = screen[j] - screen[i]\n        cx, cy, cz = screen[k] - screen[i]\n        det = bx*cy - by*cx\n        if abs(det) < 1e-9:\n            continue\n        for y in range(max(0, math.ceil(ay+min(0., by, cy))), min(height, math.floor(ay+max(0., by, cy))+1)):\n            for x in range(max(0, math.ceil(ax+min(0., bx, cx))), min(width, math.floor(ax+max(0., bx, cx))+1)):\n                u = ((x-ax)*cy-(y-ay)*cx)/det\n                v = (bx*(y-ay)-by*(x-ax))/det\n                z = az+u*bz+v*cz\n                if u >= 0 and v >= 0 and u+v <= 1 and z > depth[y,x]:\n                    owner[y,x] = fi\n                    bary[y,x,0], bary[y,x,1] = u,v\n                    depth[y,x] = z\n    return owner, bary, depth\n\n\ndef camera(vertices, yaw, elevation):\n    a,b = np.deg2rad([yaw,elevation])\n    right = np.array([np.cos(a),0,-np.sin(a)])\n    direction = np.array([np.sin(a)*np.cos(b),np.sin(b),np.cos(a)*np.cos(b)])\n    basis = np.stack([right,np.cross(right,direction),direction]).astype(\'f\')\n    projected = vertices @ basis.T\n    extent = np.maximum(np.ptp(projected[:,:2],axis=0),1e-6)\n    projected[:,:2] = (projected[:,:2]-projected[:,:2].min(0))/extent\n    return projected, direction, extent[0]/extent[1]\n\n\ndef interpolate(values, triangles, weights):\n    a,b,c = values[triangles].transpose(1,0,2)\n    return a+(b-a)*weights[:,0:1]+(c-a)*weights[:,1:2]\n\n\ndef foreground(photo, session):\n    photo = photo.copy()\n    photo.thumbnail((1280,1280),Image.Resampling.LANCZOS)\n    if photo.getchannel(\'A\').getextrema()[0] == 255:\n        from rembg import remove\n        photo = remove(photo, session=session)\n    bounds = photo.getchannel(\'A\').point(lambda a: 255 if a>128 else 0).getbbox()\n    if bounds is None or min(bounds[2]-bounds[0],bounds[3]-bounds[1]) < 8:\n        return None, (False,)*4\n    crop = np.asarray(photo.crop(bounds))\n    mask = crop[:,:,3] > 128\n    sides = (mask[:,0],mask[:,-1],mask[0],mask[-1])\n    at_edge = (bounds[0]==0,bounds[2]==photo.width,bounds[1]==0,bounds[3]==photo.height)\n    return crop, tuple(edge and side.mean()>.08 for edge,side in zip(at_edge,sides))\n\n\ndef visible_windows(edges, photo_aspect, model_aspect):\n    """Candidate regions of the projected mesh visible inside a clipped photo."""\n    left,right,top,bottom = edges\n    horizontal,vertical = left or right,top or bottom\n    if horizontal == vertical:\n        yield (0.,1.,0.,1.)\n        return\n    span = min(1., photo_aspect/model_aspect if horizontal else model_aspect/photo_aspect)\n    low,high = (left,right) if horizontal else (top,bottom)\n    starts = np.linspace(0,1-span,13) if low and high else [1-span if low else 0.]\n    for start in starts:\n        yield (float(start),span,0.,1.) if horizontal else (0.,1.,float(start),span)\n\n\ndef joint_cameras(scores, labels):\n    offsets = {\'front\': 0, \'right\': -90, \'rear\': 180, \'left\': 90}\n\n    def allowed(label, front, candidate):\n        yaw,elev = candidate[1:3]\n        label,_,nose = label.partition(\'_nose_\')\n        signed = (yaw-front-offsets.get(label,0)+180)%360-180\n        delta = abs(signed)\n        if nose==\'left\' and signed<=0 or nose==\'right\' and signed>=0:\n            return False\n        if label in offsets:\n            return delta <= 15 and abs(elev) <= 12\n        if label in (\'top\',\'bottom\'):\n            return elev >= 70 if label == \'top\' else elev <= -70\n        if label.endswith(\'three_quarter\'):\n            return (20 <= delta <= 70 if label.startswith(\'front\') else 110 <= delta <= 160) and abs(elev) <= 50\n        return True\n\n    # The generated object\'s forward axis is unknown. Score complete assignments\n    # around one shared heading, including the side profiles that reveal its nose.\n    best = None\n    for front in range(0,360,30):\n        picks = [max((c for c in row if allowed(label,front,c)), default=None)\n                 for row,label in zip(scores,labels)]\n        quality = sum(p[0] if p else -1 for p in picks)\n        if best is None or quality > best[0]:\n            best = quality,picks\n    return best[1]\n\n\ndef refine(mesh, images, labels=None):\n    mat = mesh.visual.material\n    atlas = np.asarray(mat.baseColorTexture.convert(\'RGB\'))\n    h,w = atlas.shape[:2]\n    uv = np.asarray(mesh.visual.uv,dtype=\'f\').copy()\n    uv[:,1] = 1-uv[:,1]\n    uv *= [w-1,h-1]\n    vertices = np.asarray(mesh.vertices,dtype=\'f\')\n    faces = np.asarray(mesh.faces,dtype=np.int32)\n    session = None\n    if any(im.getchannel(\'A\').getextrema()[0] == 255 for im in images):\n        from rembg import new_session\n        session = new_session(\'u2net\', providers=[\'CPUExecutionProvider\'])\n    photos,edges = zip(*(foreground(im, session) for im in images))\n    del session\n    refs = [None if p is None else cv2.resize(p,(64,64),interpolation=cv2.INTER_AREA) for p in photos]\n    scores = [[] for _ in photos]\n    grid_y,grid_x = np.indices((64,64),dtype=\'f\')\n\n    def evaluate(yaw,elev,indices):\n        screen,_,aspect = camera(vertices,yaw,elev)\n        screen[:,:2] *= 63\n        face,bary,_ = raster(screen,faces,64,64)\n        hit = face>=0\n        tex = interpolate(uv,faces[face[hit]],bary[hit]).round().astype(int)\n        render = np.zeros((64,64,3),np.float32)\n        render[hit] = atlas[tex[:,1].clip(0,h-1),tex[:,0].clip(0,w-1)]\n        for i in indices:\n            ref = refs[i]\n            if ref is None:\n                continue\n            mask = ref[:,:,3]>128\n            ratio = photos[i].shape[1]/photos[i].shape[0]\n            best = None\n            for window in visible_windows(edges[i],ratio,aspect):\n                sx,sw,sy,sh = window\n                if sw==sh==1:\n                    visible,colors = hit,render\n                else:\n                    mx,my = grid_x*sw+63*sx,grid_y*sh+63*sy\n                    visible = cv2.remap(hit.astype(\'uint8\'),mx,my,cv2.INTER_NEAREST)>0\n                    colors = cv2.remap(render,mx,my,cv2.INTER_LINEAR)\n                overlap = mask & visible\n                if not overlap.any():\n                    continue\n                iou = overlap.sum()/(mask|visible).sum()\n                target = ref[:,:,:3][overlap].astype(\'f\')\n                projected = colors[overlap]\n                gain = np.clip((target.mean()+1)/(projected.mean()+1),.6,1.7)\n                error = np.abs(target-projected*gain).mean()/255\n                aspect_error = abs(np.log(aspect*sw/(ratio*sh)))\n                score = iou-.72*error-.18*min(aspect_error,2)\n                candidate = (score,yaw,elev,iou,error,window)\n                if best is None or score>best[0]:\n                    best = candidate\n            if best is not None:\n                scores[i].append(best)\n\n    for elev in (-90,-45,-20,0,20,45,90):\n        for yaw in range(0,360,30):\n            evaluate(yaw,elev,range(len(refs)))\n    supplied_labels = labels or [\'unknown\']*len(images)\n    labels = list(supplied_labels)\n    for i,(ref,label) in enumerate(zip(refs,labels)):\n        if ref is None or \'three_quarter\' not in label:\n            continue\n        # Cross-check oblique predictions on bilaterally symmetric end views.\n        gray = cv2.cvtColor(ref[:,:,:3],cv2.COLOR_RGB2GRAY).astype(\'f\')\n        mask = ref[:,:,3]>128\n        overlap = mask & mask[:,::-1]\n        values, mirror = gray[overlap],gray[:,::-1][overlap]\n        if (overlap.sum() > .87*mask.sum() and values.size and values.std()>5\n                and np.corrcoef(values,mirror)[0,1]>.72):\n            labels[i] = label.split(\'_\')[0]\n    for i,match in enumerate(joint_cameras(scores,labels)):\n        if match:\n            a,b = match[1:3]\n            for da in (-12,0,12):\n                for db in (-8,0,8):\n                    evaluate(a+da,np.clip(b+db,-90,90),[i])\n\n    owner,bary,_ = raster(np.column_stack([uv,np.zeros(len(uv),\'f\')]),faces,h,w)\n    normals = np.asarray(mesh.vertex_normals,dtype=\'f\')\n    weighted = np.zeros((h,w,3),\'f\')\n    weights = np.zeros((h,w),\'f\')\n    report = []\n    for pic,match,label,candidates in zip(photos,joint_cameras(scores,labels),labels,scores):\n        if not match:\n            report.append(dict(applied=False,label=label,reason=\'no eligible camera\'))\n            continue\n        score,yaw,elev,iou,error,window = match\n        if label in (\'unknown\',\'oblique\') and abs(elev)<65 and any(\n            abs((other[1]-yaw+180)%360-180)>140 and abs(other[2]-elev)<25\n            and other[0]>=score-.025 for other in candidates\n        ):\n            report.append(dict(applied=False,label=label,reason=\'ambiguous opposite directions\'))\n            continue\n        applied = bool(iou>=.70 and score>=.52 and error<.36)\n        report.append(dict(applied=applied,label=label,yaw=float(yaw),elevation=float(elev),silhouette_iou=round(float(iou),3),score=round(float(score),3)))\n        if not applied:\n            continue\n        ph,pw = pic.shape[:2]\n        projected,direction,_ = camera(vertices,yaw,elev)\n        sx,sw,sy,sh = window\n        projected[:,:2] = (projected[:,:2]-[sx,sy])/[sw,sh]\n        report[-1][\'visible_window\'] = list(window)\n        projected[:,:2] *= [pw-1,ph-1]\n        _,_,depth = raster(projected,faces,ph,pw)\n        alpha = cv2.erode(pic[:,:,3],np.ones((3,3),np.uint8))\n        tolerance = np.linalg.norm(np.ptp(vertices,axis=0))*.009\n        for row in range(0,h,64):\n            block = owner[row:row+64]\n            yy,xx = np.where(block>=0)\n            yy += row\n            tri = faces[owner[yy,xx]]\n            frac = bary[yy,xx]\n            q = interpolate(projected,tri,frac)\n            px,py = np.rint(q[:,:2]).astype(int).T\n            inside = (px>=0)&(px<pw)&(py>=0)&(py<ph)\n            px,py = px.clip(0,pw-1),py.clip(0,ph-1)\n            depth_ok = inside & (np.abs(depth[py,px]-q[:,2])<tolerance)\n            facing = interpolate(normals,tri,frac) @ direction\n            weight = np.maximum(facing,0)**8 * depth_ok * (alpha[py,px]>225) * score**4\n            weighted[yy,xx] += weight[:,None]*pic[py,px,:3]\n            weights[yy,xx] += weight\n    mix = np.minimum(weights/.065,1)\n    fused = atlas*(1-mix[:,:,None])+weighted*(mix/np.maximum(weights,1e-10))[:,:,None]\n    mat.baseColorTexture = Image.fromarray(np.uint8(np.clip(fused,0,255)))\n    for item,label in zip(report,supplied_labels):\n        item[\'supplied_label\'] = label\n    return dict(reference_registration=report,photo_texture_fraction=round(float(np.count_nonzero(mix>.5)/max(1,(owner>=0).sum())),4))\n\n\ndef unlit(document):\n    document.setdefault(\'extensionsUsed\',[]).append(\'KHR_materials_unlit\')\n    for material in document[\'materials\']:\n        material.setdefault(\'extensions\',{})[\'KHR_materials_unlit\'] = {}\n'

if 'studio_proc' in globals() and studio_proc.poll() is None:
    raise RuntimeError('The service is already running. Run the final cell to copy its URL, or restart the runtime for changed code.')
STUDIO_KEY=secrets.token_urlsafe(32)
path=pathlib.Path('/content/TRELLIS/form_service.py')
path.write_text(service_source)
path.with_name('photo_finish.py').write_text(photo_source)
env=os.environ.copy()
env.update(STUDIO_KEY=STUDIO_KEY, PATH='/content/trellis-env/bin:'+env['PATH'], CUDA_HOME='/content/trellis-env', CC='/content/trellis-env/bin/x86_64-conda-linux-gnu-cc', CXX='/content/trellis-env/bin/x86_64-conda-linux-gnu-c++', ATTN_BACKEND='xformers', SPCONV_ALGO='native', TORCH_CUDA_ARCH_LIST='8.0', MAX_JOBS='4')
env['CPATH']=':'.join(['/content/trellis-env/targets/x86_64-linux/include']+[str(p) for p in pathlib.Path('/content/trellis-env/lib/python3.10/site-packages/nvidia').glob('*/include')])
env['LIBRARY_PATH']='/content/trellis-env/targets/x86_64-linux/lib'
log=open('/content/form-service.log','w')
studio_proc=subprocess.Popen(['/content/trellis-env/bin/python','-u',str(path)],cwd='/content/TRELLIS',env=env,stdout=log,stderr=subprocess.STDOUT)
for attempt in range(600):
    if studio_proc.poll() is not None:
        raise RuntimeError(pathlib.Path('/content/form-service.log').read_text()[-6000:])
    try:
        request=urllib.request.Request('http://127.0.0.1:7860/health',headers={'Authorization':'Bearer '+STUDIO_KEY})
        with urllib.request.urlopen(request,timeout=2) as response:
            if response.status==200:break
    except Exception:time.sleep(2)
else:raise TimeoutError('Loading is taking longer than expected. Inspect /content/form-service.log before restarting.')
print('TRELLIS ready. Run the final cell to get your connection URL.')


In [ ]:
# @title 3. Copy this connection URL into localhost
import pathlib, subprocess, urllib.request, re, time
if 'studio_proc' not in globals() or studio_proc.poll() is not None:
    raise RuntimeError('Run the service cell first.')
if 'tunnel_proc' not in globals() or tunnel_proc.poll() is not None:
    binary=pathlib.Path('/content/cloudflared')
    if not binary.exists():
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',binary)
        binary.chmod(0o755)
    tunnel_log=open('/content/form-tunnel.log','w')
    tunnel_proc=subprocess.Popen([str(binary),'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
for attempt in range(90):
    if tunnel_proc.poll() is not None:raise RuntimeError('Tunnel stopped; inspect /content/form-tunnel.log')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',pathlib.Path('/content/form-tunnel.log').read_text())
    if match:break
    time.sleep(1)
else:raise TimeoutError('Tunnel URL not ready; rerun this cell.')
print('Paste this entire URL into MeshPort (keep it private):')
print(match.group(0)+'/#key='+STUDIO_KEY)
print('Keep this Colab runtime connected. Reconnecting a new runtime creates a new URL.')
